In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, glob, os
import scipy.io as sio

## Objectives
1. Create preQC_df, do QC offline, load QC_df, store various cluster/unit/neuron figs in different directories
2. Create neur_df with columns for spikes, region, coordinates
3. Merge clusters as needed
4. Inspect & save

### variables

In [2]:
patient = 18
save = False

units_dir = f'../../data/units'
units_all_dir = f'{units_dir}/all/2025{patient}'
units_waves_dir = f'{units_dir}/waves/2025{patient}'
units_clean_dir = f'{units_dir}/clean/2025{patient}'
units_maybes_dir = f'{units_dir}/maybes/2025{patient}'
for dir in [units_all_dir, units_waves_dir, units_clean_dir, units_maybes_dir]:
    os.makedirs(dir, exist_ok=True)

## 1. QC stuff

In [3]:
# save = True

# parse osort figs
for file in glob.glob(f'../../results/2025{patient}/osort_mat/figs/5/*'):

    # grab individual CL figs
    if 'CL' in os.path.basename(file) and 'ALL' not in os.path.basename(file):
        
        dest = os.path.join(units_all_dir, os.path.basename(file))
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {file} {units_all_dir}/')

    # grab WAVES figs to check for mergers
    if 'WAVES' in os.path.basename(file):
        
        dest = os.path.join(units_waves_dir, os.path.basename(file))
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {file} {units_waves_dir}/')

save = False # reset to be safe

### create preQC_df
note: clustID called unitID going forward

In [4]:
# init preQC_df 
preQC_df = pd.DataFrame(columns=['chanID', 'unitID'])
chanIDs, unitIDs = [], []

# parse units_all_dir
for file in glob.glob(f'{units_all_dir}/*'):
    chanIDs.append(int(os.path.basename(file).split('_')[0][1:]))
    unitIDs.append(int(os.path.basename(file).split('_')[2]))

# create df
preQC_df = pd.DataFrame({'chanID': chanIDs, 'unitID': unitIDs})
preQC_df = preQC_df.sort_values(by=['chanID', 'unitID']).reset_index(drop=True)

# save df
preQC_df_path = f'../../results/2025{patient}/records/preQC_pt{patient}.csv'
if not os.path.exists(preQC_df_path):
    print(f'Saving preQC_df to {preQC_df_path}')
    preQC_df.to_csv(preQC_df_path, index=False)

save = False # reset to be safe
preQC_df

,chanID,unitID
0,193,421
1,193,467
2,194,179
3,196,308
4,197,203
...,...,...
65,221,3
66,222,4
67,223,2
68,223,3


### load QC, and also save QC neur figs in units/clean/{patient} and units/maybes/{patient}

In [5]:
# save=True

QC_df = pd.read_csv(f'../../results/2025{patient}/records/QC_pt{patient}.csv')
clean_df = QC_df[QC_df['keep'] == 1].copy().reset_index(drop=True)
maybe_df = QC_df[~QC_df['notes'].isna()].copy().reset_index(drop=True)

for unit_file in glob.glob(f'{units_all_dir}/*'):
    unitID = int(os.path.basename(unit_file).split('_')[2])
    
    if unitID in clean_df['unitID'].tolist():
        dest = f'{units_clean_dir}/{os.path.basename(unit_file)}'
        
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {unit_file} {dest}')
    
    if unitID in maybe_df['unitID'].tolist():
        dest = f'{units_maybes_dir}/{os.path.basename(unit_file)}'
        
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {unit_file} {dest}')

assert len(clean_df) == len(glob.glob(f'{units_clean_dir}/*.png')), f'Length mismatch: clean_df has {len(clean_df)} rows, but {len(glob.glob(f"{units_clean_dir}/*.png"))} files in {units_clean_dir}'

save = False # reset to be safe
clean_df

,chanID,unitID,keep,notes
0,193,467,1.0,NaN
1,202,871,1.0,NaN
2,204,701,1.0,NaN
3,204,708,1.0,NaN
4,205,590,1.0,NaN
5,206,612,1.0,NaN
6,207,1988,1.0,NaN
7,210,4424,1.0,NaN
8,210,4457,1.0,NaN
9,210,4546,1.0,NaN


## 2. neur_df stuff

### helpers

In [6]:
def getunitID2spikes(unitIDs, spikes, clean_df):
    ''' return dict with keys=unique units, and vals = list of corresponding spikes '''
    
    # initialize unit2spikes with keys as unique unit IDs and empty list as values
    unit2spikes = {}
    for unitID, spike in zip(unitIDs, spikes):

        if unitID not in clean_df['unitID'].tolist(): continue
        if unitID not in unit2spikes: unit2spikes[unitID] = [] # initialize

        unit2spikes[unitID].append(spike)

    return unit2spikes

### First, add spike data from OSort mats.

In [7]:
samp_rate = 1000000
# columns
chanID_list, unitID_list, spikes_list, num_spikes_list, FR_list = [], [], [], [], []

# go through OSort mat files
for mat_file in glob.glob(f'../../results/2025{patient}/osort_mat/sorted_mats/5/*_sorted_new.mat'):

    # load channel mat
    chanMat = sio.loadmat(mat_file)
    chanID = int(os.path.basename(mat_file).split('_')[0][1:])

    # load vectors of spike times and corresponding units
    if chanMat['assignedNegative'].size == 0: continue
    spike_vector = chanMat['newTimestampsNegative'][0] # len = total n_spikes
    unit_vector = chanMat['assignedNegative'][0] # len = total n_spikes

    # create unit_vector => [spike_vector] for QCed units
    unit2spikes = getunitID2spikes(unit_vector, spike_vector, clean_df)
    for unitID, unit_spike_list in unit2spikes.items():
        chanID_list.append(chanID)
        unitID_list.append(unitID)
        spikes = np.array(unit_spike_list) / samp_rate  # seconds
        spikes_list.append(spikes)
        num_spikes_list.append(len(spikes))
        FR_list.append(len(spikes) / (spikes[-1] - spikes[0]))

neur_df = pd.DataFrame({'chanID': chanID_list, 'unitID': unitID_list, 'spikes': spikes_list, 'num_spikes': num_spikes_list, 'FR': FR_list})
assert len(neur_df) == len(clean_df), f'Length mismatch: neur_df has {len(neur_df)} rows, clean_df has {len(clean_df)} rows'
neur_df


,chanID,unitID,spikes,num_spikes,FR
0,210,4424,"[0.21733333333333335, 0.40853333333333336, 0.5...",9533,5.760531
1,210,4546,"[0.3117666666666667, 0.3276333333333334, 0.340...",6376,3.833812
2,210,4457,"[0.8058666666666667, 11.403633333333334, 17.65...",5651,3.417072
3,207,1988,"[0.25660000000000005, 0.5228666666666667, 0.72...",3825,2.299444
4,216,1110,"[2.805766666666667, 3.6407333333333334, 4.8141...",750,0.451857
5,205,590,"[3.493566666666667, 4.5576, 7.251833333333334,...",912,0.552273
6,202,871,"[3.734466666666667, 4.718, 8.183800000000002, ...",1114,0.681801
7,206,612,"[0.017533333333333335, 0.6868666666666667, 3.7...",951,0.576399
8,193,467,"[3.260966666666667, 5.094733333333334, 11.0821...",1250,0.757284
9,204,701,"[0.5282, 1.0729333333333335, 1.3612, 2.3284666...",1727,1.043810


### add region column by mapping channel -> region (label)

In [8]:
chanMap = sio.loadmat(glob.glob(f'../../results/2025{patient}/records/*ChannelMap*.mat')[0])

# variable across patients
if patient in [12, 21]: channelMap = chanMap['ChannelMap1'].flatten()
elif patient == 18: channelMap = chanMap['ChannelMap2'].flatten()

labelMap = chanMap['LabelMap'].flatten() # contains region labels
labelMap = np.array([str(label.squeeze()) for label in labelMap])  # convert to str

nan_mask = ~np.isnan(channelMap)
channel2label = dict(zip(channelMap[nan_mask], labelMap[nan_mask]))

neur_df['region'] = neur_df['chanID'].map(channel2label).fillna(neur_df['chanID']).apply(lambda x: str(x))
neur_df

,chanID,unitID,spikes,num_spikes,FR,region
0,210,4424,"[0.21733333333333335, 0.40853333333333336, 0.5...",9533,5.760531,mRAHIP2
1,210,4546,"[0.3117666666666667, 0.3276333333333334, 0.340...",6376,3.833812,mRAHIP2
2,210,4457,"[0.8058666666666667, 11.403633333333334, 17.65...",5651,3.417072,mRAHIP2
3,207,1988,"[0.25660000000000005, 0.5228666666666667, 0.72...",3825,2.299444,mRACC7
4,216,1110,"[2.805766666666667, 3.6407333333333334, 4.8141...",750,0.451857,mRAHIP8
5,205,590,"[3.493566666666667, 4.5576, 7.251833333333334,...",912,0.552273,mRACC5
6,202,871,"[3.734466666666667, 4.718, 8.183800000000002, ...",1114,0.681801,mRACC2
7,206,612,"[0.017533333333333335, 0.6868666666666667, 3.7...",951,0.576399,mRACC6
8,193,467,"[3.260966666666667, 5.094733333333334, 11.0821...",1250,0.757284,mROFC1
9,204,701,"[0.5282, 1.0729333333333335, 1.3612, 2.3284666...",1727,1.043810,mRACC4


### add coordinates columns by mapping region->coords

In [9]:
# cleaning function to handle nested arrays and bytes
def clean_entry(x):
    while isinstance(x, (np.ndarray, list)):
        x = x[0]
    if isinstance(x, (bytes, bytearray)):
        x = x.decode("utf-8", errors="ignore")
    return str(x)

at some point, figure out this code line by line

In [10]:
# load
electrodeInfo = sio.loadmat(glob.glob(
                f'../../results/2025{patient}/records/*DI_Electrodes*.mat')[0])
ElecMapRaw   = pd.DataFrame(electrodeInfo['ElecMapRaw']) # region -> ID
ElecXYZRaw   = pd.DataFrame(electrodeInfo['ElecXYZRaw']) # ID -> coordinates
ElecAtlasRaw = pd.DataFrame(electrodeInfo['ElecAtlasRaw']) # atlas coords?

atlas_index = 0 # NMM

# clean
region_s = ElecMapRaw[0].apply(clean_entry)                    # Series of regions
atlas_s  = ElecAtlasRaw.iloc[:, atlas_index].apply(clean_entry)  # Series of atlas regions

# build small tables
region2id_df = pd.DataFrame({"region": region_s.values,
                             "ID": np.arange(len(region_s))})
id2xyz_df = ElecXYZRaw.reset_index().rename(columns={'index':'ID', 0:'x', 1:'y', 2:'z'})
# xyz2atlasRegions = pd.DataFrame({"ID": np.arange(len(atlas_s)),
#                                  "atlas_region": atlas_s.values})

# merge
final_neur_df = (neur_df
                .merge(region2id_df, on='region', how='left')
                .merge(id2xyz_df, on='ID', how='left')
                # .merge(xyz2atlasRegions, on='ID', how='left')
                )
final_neur_df = final_neur_df.drop(columns=['ID'])
final_neur_df['spikes'] = final_neur_df['spikes'].apply(lambda x: np.array(x))  # ensure arrays

# sort by chanID
final_neur_df = final_neur_df.sort_values(by=['chanID', 'unitID']).reset_index(drop=True)
final_neur_df

,chanID,unitID,spikes,num_spikes,FR,region,x,y,z
0,193,467,"[3.260966666666667, 5.094733333333334, 11.0821...",1250,0.757284,mROFC1,5.400004,42.999992,-10.400004
1,202,871,"[3.734466666666667, 4.718, 8.183800000000002, ...",1114,0.681801,mRACC2,6.600004,29.799992,23.199994
2,204,701,"[0.5282, 1.0729333333333335, 1.3612, 2.3284666...",1727,1.043810,mRACC4,7.800004,27.399993,23.199994
3,204,708,"[5.711866666666667, 12.2751, 13.62493333333333...",1026,0.622157,mRACC4,7.800004,27.399993,23.199994
4,205,590,"[3.493566666666667, 4.5576, 7.251833333333334,...",912,0.552273,mRACC5,7.800004,29.799992,23.199994
5,206,612,"[0.017533333333333335, 0.6868666666666667, 3.7...",951,0.576399,mRACC6,9.000004,28.599993,23.199994
6,207,1988,"[0.25660000000000005, 0.5228666666666667, 0.72...",3825,2.299444,mRACC7,9.000004,27.399993,23.199994
7,210,4424,"[0.21733333333333335, 0.40853333333333336, 0.5...",9533,5.760531,mRAHIP2,16.600004,-3.000006,-16.400003
8,210,4457,"[0.8058666666666667, 11.403633333333334, 17.65...",5651,3.417072,mRAHIP2,16.600004,-3.000006,-16.400003
9,210,4546,"[0.3117666666666667, 0.3276333333333334, 0.340...",6376,3.833812,mRAHIP2,16.600004,-3.000006,-16.400003


### 3. Merge clusters as needed
1. youd need to make a merge col to merge in 'unclean' units. retain this col alongside the 'keep' col.

In [ ]:
# grab rows to merge
merge_rows_idx = (final_neur_df['chanID'] == 204) & (final_neur_df['unitID'].isin([701, 708]))
merge_rows = final_neur_df[merge_rows_idx]

# merge spikes
merged_spikes = np.sort(np.concatenate(merge_rows['spikes'].values))

# retain & update row values as needed
merged_row = merge_rows.iloc[0].copy()
merged_row['unitID'], merged_row['spikes'], merged_row['num_spikes'], merged_row['FR'] =\
'701/708',            merged_spikes,        len(merged_spikes),       len(merged_spikes) / (merged_spikes[-1] - merged_spikes[0])

tst = pd.concat([final_neur_df[~merge_rows_idx], merged_row.to_frame().T], ignore_index=True)
tst = tst.sort_values(by=['chanID']).reset_index(drop=True)
tst

,patient,chanID,unitID,spikes,num_spikes,FR,region,x,y,z
0,18,193,467,"[3.260966666666667, 5.094733333333334, 11.0821...",1250,0.757284,mROFC1,5.400004,42.999992,-10.400004
1,18,202,871,"[3.734466666666667, 4.718, 8.183800000000002, ...",1114,0.681801,mRACC2,6.600004,29.799992,23.199994
2,18,204,701/708,"[0.5282, 1.0729333333333335, 1.3612, 2.3284666...",2753,1.663931,mRACC4,7.800004,27.399993,23.199994
3,18,205,590,"[3.493566666666667, 4.5576, 7.251833333333334,...",912,0.552273,mRACC5,7.800004,29.799992,23.199994
4,18,206,612,"[0.017533333333333335, 0.6868666666666667, 3.7...",951,0.576399,mRACC6,9.000004,28.599993,23.199994
5,18,207,1988,"[0.25660000000000005, 0.5228666666666667, 0.72...",3825,2.299444,mRACC7,9.000004,27.399993,23.199994
6,18,210,4424,"[0.21733333333333335, 0.40853333333333336, 0.5...",9533,5.760531,mRAHIP2,16.600004,-3.000006,-16.400003
7,18,210,4457,"[0.8058666666666667, 11.403633333333334, 17.65...",5651,3.417072,mRAHIP2,16.600004,-3.000006,-16.400003
8,18,210,4546,"[0.3117666666666667, 0.3276333333333334, 0.340...",6376,3.833812,mRAHIP2,16.600004,-3.000006,-16.400003
9,18,211,2862,"[2.716666666666667, 2.723366666666667, 2.73293...",1817,1.094763,mRAHIP3,16.600004,-0.600006,-16.400003


### 4. inspect & save

In [11]:
print('example neuron')
print(f'last 5 spikes (s): {final_neur_df["spikes"].iloc[0][-5:]}')
print(f'last 5 spikes (min): {final_neur_df["spikes"].iloc[0][-5:]/60}')

# add patient as first column
if 'patient' not in final_neur_df: final_neur_df.insert(0, 'patient', patient)

# make dir
processed_data_dir = f'../../results/2025{patient}/records/processed_data'
os.makedirs(processed_data_dir, exist_ok=True)

# save file
parquet_path = os.path.join(processed_data_dir, 'df_neurs.parquet')
if os.path.exists(parquet_path):     print(f'File already exists, skipping: {parquet_path}')
else:     final_neur_df.to_parquet(parquet_path, index=False)
# final_neur_df.to_parquet(parquet_path, index=False)

example neuron
last 5 spikes (s): [1639.3013 1640.7237 1642.3562 1652.0312 1653.8973]
last 5 spikes (min): [27.32168833 27.345395   27.37260333 27.53385333 27.564955  ]
File already exists, skipping: ../../results/202518/records/processed_data/df_neurs.parquet
